In [7]:
%ls ../../

Invalid switch - "..".


In [13]:
import pandas as pd

df_cuenta_corriente_historico = pd.read_csv(
    '../data/analytics/cuentas_unificadas_usd_sorted.csv'
)

df_cuenta_corriente_historico.columns

Index(['Liquida', 'Operado', 'Comprobante', 'Numero', 'Cantidad', 'Especie',
       'Precio', 'Importe', 'Saldo', 'Referencia', 'Origen'],
      dtype='str')

In [ ]:
{
    "RECIBO DE COBRO": "Aumenta el valor en Cash_Total_USD",
    "SUSCRIPCION FCI": "Disminuye el valor en Cash_Total_USD y lo aumenta en Total_Growth_Valuation",
    "COMPRA NORMAL": "Disminuye el valor en Cash_Total_USD y lo aumenta en Total_Growth_Valuation o Total_Safe_Valuation",
    "ORDEN DE PAGO": "Disminuye el valor en Cash_Total_USD",
    "LIQUIDACION RESCATE FCI": "Aumenta el valor en Cash_Total_USD y lo disminuye en Total_Growth_Valuation",
    "VENTA": "Aumenta el valor en Cash_Total_USD y disminuye en Total_Growth_Valuation o Total_Safe_Valuation",
    "ORD PAGO DOLARES": "Disminuye el valor en Cash_Total_USD",
    "VENTA PARIDAD": "Aumenta el valor en Cash_Total_USD y disminuye en Total_Growth_Valuation o Total_Safe_Valuation",
    "REC COBRO DOLARES": "Aumenta el valor en Cash_Total_USD",
    "COMPRA PARIDAD": "Disminuye el valor en Cash_Total_USD y lo aumenta en Total_Growth_Valuation o Total_Safe_Valuation",
    "PAGO DIV": "Aumenta el valor en Cash_Total_USD",
    "DIVIDENDOS": "Aumenta el valor en Cash_Total_USD",
    "CREDITO DERMERC-": "Aumenta el valor en Cash_Total_USD",
    "COMPRA TRADING": "Disminuye el valor en Cash_Total_USD y lo aumenta en Total_Growth_Valuation o Total_Safe_Valuation",
    "VENTA TRADING": "Aumenta el valor en Cash_Total_USD y disminuye en Total_Growth_Valuation o Total_Safe_Valuation",
    "RETENCION": "Disminuye el valor en Cash_Total_USD",
    "LICITACION PARIDAD": "Disminuye el valor en Cash_Total_USD y lo aumenta en Total_Growth_Valuation o Total_Safe_Valuation",
    "VENTA EXTERIOR V": "Aumenta el valor en Cash_Total_USD y disminuye en Total_Growth_Valuation o Total_Safe_Valuation",
    "RETENCION DOLARES": "Disminuye el valor en Cash_Total_USD",
    "DIVIDENDO TESORO": "Aumenta el valor en Cash_Total_USD",
    "COMPRA EXTERIOR V": "Disminuye el valor en Cash_Total_USD y lo aumenta en Total_Growth_Valuation o Total_Safe_Valuation",
    "NOTA DE CREDITO U$S": "Aumenta el valor en Cash_Total_USD",
    "NOTA DE DEBITOS U$S": "Disminuya el valor en Cash_Total_USD",
    "RETENCION DOLAR MEP": "Disminuya el valor en Cash_Total_USD",
    "COMPRA CAUCION CONTADO": "Disminuya el valor en Cash_Total_USD y lo aumenta en Total_Safe_Valuation",
    "VENTA CAUCION TERMINO": "Aumenta el valor en Cash_Total_USD y lo disminuye en Total_Safe_Valuation",
    "LICITACION PRIVADA": "Disminuye el valor en Cash_Total_USD y lo aumenta en Total_Growth_Valuation o Total_Safe_Valuation",
    "RENTA Y AMORTIZ": "Aumenta el valor en Cash_Total_USD"
}

In [19]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

class VectorizedVCPEngine:
    """
    Motor Mark-to-Market Vectorizado.
    Construye las matrices de Tenencias (T x N) y Precios (T x N) para calcular 
    el Valor Cuotaparte con asimetría positiva, aislando el riesgo Barbell.
    """
    def __init__(self, csv_path: str):
        self.csv_path = csv_path
        
        # Identificadores Estratégicos
        self.fcis = ['ALGIIA', 'BMACTAA', 'BULL-IA', 'BULMAAA', 'RIGAHOR']
        self.safe_bonds = ['AL30', 'GD30', 'GD35', 'AE38', 'SNSBO', 'LK01Q', 'AL30D', 'GD30D']
        
        # Universo de CEDEARs normalizados a Full Shares
        self.ratios_cedear = {
            'KO': 5.0, 'SPY': 20.0, 'QQQ': 20.0, 'AAPL': 10.0,
            'GOOGL': 58.0, 'MSFT': 30.0, 'TSLA': 15.0, 'MELI': 120.0,
            'LLY': 56.0, 'META': 24.0, 'VIST': 3.0, 'AMZN': 144.0,
            'NVDA': 24.0, 'NFLX': 60.0, 'TLT': 1.0, 'SH': 8.0,
            'ARGT': 1.0, 'XLP': 1.0, 'SHY': 1.0, 'ADBE': 44.0,
            'ARKK': 10.0, 'ASML': 146.0, 'BBD': 1.0, 'BIOX': 1.0,
            'COIN': 27.0, 'ERIC': 2.0, 'HMY': 1.0, 'LAR': 1.0,
            'PAAS': 3.0, 'PSQ': 8.0, 'SAN': 0.25, 'UNH': 33.0, 'VALE': 2.0
        }

    def _get_engine(self):
        env_path = r'c:\Users\tomas\white_finance\.env'
        if os.path.exists(env_path):
            load_dotenv(env_path)
            
        user = os.getenv("POSTGRE_USER", "postgres")
        pwd = os.getenv("POSTGRE_PASSWORD", "postgres")
        host = os.getenv("POSTGRE_HOST", "localhost")
        port = os.getenv("POSTGRE_PORT", "5432")
        db = os.getenv("POSTGRE_DB", "postgres")
        return create_engine(f"postgresql+psycopg2://{user}:{pwd}@{host}:{port}/{db}")

    def _fetch_market_data(self):
        """Extrae la base de precios y el tipo de cambio unificado desde PostgreSQL"""
        engine = self._get_engine()
        query = """
        SELECT hp.date, ticker, 
            case when "source" <> 'YFinance_USD' then "close" / ccl else "close" end as close_usd,
            "source",
            ccl
        FROM earnings.historical_prices hp
        left join earnings.ccl_mep cm 
        on hp.date = cm."date";
        """
        df_px = pd.read_sql(query, engine)
        df_px['date'] = pd.to_datetime(df_px['date'])
        
        # Extraemos la serie del CCL limpia para dolarizaciones dinámicas
        df_ccl = df_px[['date', 'ccl']].dropna().drop_duplicates().set_index('date')['ccl']
        return df_px, df_ccl

    def _preprocess_trades(self):
        """Carga y limpia el dataset transaccional generando los flujos de cantidades"""
        df_trades = pd.read_csv(self.csv_path)
        df_trades['Operado'] = pd.to_datetime(df_trades['Operado'])
        
        # Generar signos vectorizados para el Flujo de Cantidad (Stock)
        ventas_comprobantes = [
            'VENTA', 'VENTA EXTERIOR V', 'VENTA CAUCION TERMINO', 
            'VENTA PARIDAD', 'RESCATE FCI'
        ]
        df_trades['Signo_Cant'] = df_trades['Comprobante'].apply(lambda x: -1 if x in ventas_comprobantes else 1)
        df_trades['Cantidad'] = df_trades['Cantidad'].fillna(0.0)
        df_trades['Flujo_Cant'] = df_trades['Cantidad'] * df_trades['Signo_Cant']
        
        return df_trades

    def build_portfolio_metrics(self):
        print("1. Preprocesando Transacciones y Datos de Mercado...")
        df_trades = self._preprocess_trades()
        df_prices_sql, series_ccl = self._fetch_market_data()
        
        # Grilla Temporal Continua
        date_range = pd.date_range(start=df_trades['Operado'].min(), end=pd.Timestamp.today().normalize())
        series_ccl = series_ccl.reindex(date_range).ffill()
        
        print("2. Procesando Componentes Safe Base (Liquidez, FCIs y Cauciones)...")
        # --- CASH LÍQUIDO (Saldo Histórico) ---
        # Como los saldos ya están en USD CCL, tomamos el último saldo del día por cada Origen
        df_saldos = df_trades.dropna(subset=['Saldo'])
        df_cash_pivot = df_saldos.groupby(['Operado', 'Origen'])['Saldo'].last().unstack().ffill()
        df_cash_grid = df_cash_pivot.reindex(date_range).ffill().fillna(0)
        cash_total_usd = df_cash_grid.sum(axis=1)

        # --- FCIs (Saldos Sintéticos) ---
        df_fci = df_trades[df_trades['Especie'].isin(self.fcis)].copy()
        # El Importe está en USD CCL de la fecha operada. Lo revertimos a ARS nominal y lo acumulamos.
        df_fci['ccl_operacion'] = df_fci['Operado'].map(series_ccl)
        df_fci['Flujo_ARS'] = (df_fci['Importe'].fillna(0) * df_fci['ccl_operacion']) * -1 
        fci_grid_ars = df_fci.groupby('Operado')['Flujo_ARS'].sum().reindex(date_range).fillna(0).cumsum()
        fci_grid_usd = fci_grid_ars / series_ccl # Mark-to-Market real al CCL del día

        # --- CAUCIONES ---
        df_cauc = df_trades[df_trades['Comprobante'].str.contains('CAUCION', na=False)].copy()
        df_cauc['ccl_operacion'] = df_cauc['Operado'].map(series_ccl)
        df_cauc['Flujo_Caucion_ARS'] = (df_cauc['Importe'].fillna(0) * df_cauc['ccl_operacion']) * -1
        cauc_grid_ars = df_cauc.groupby('Operado')['Flujo_Caucion_ARS'].sum().reindex(date_range).fillna(0).cumsum()
        cauc_grid_usd = cauc_grid_ars / series_ccl

        print("3. Construyendo Matrices Vectorizadas de Equities...")
        # --- MATRIZ DE TENENCIAS (T x N) ---
        # Filtramos activos transaccionales que cotizan en mercado (excluimos FCIs y Cauciones)
        filtro_mercado = (~df_trades['Especie'].isin(self.fcis)) & (~df_trades['Comprobante'].str.contains('CAUCION', na=False)) & (df_trades['Especie'].notna())
        df_mkt = df_trades[filtro_mercado].copy()
        
        holdings = df_mkt.groupby(['Operado', 'Especie'])['Flujo_Cant'].sum().unstack(fill_value=0)
        df_holdings_grid = holdings.reindex(date_range).fillna(0).cumsum()
        
        # --- MATRIZ DE PRECIOS (T x N) ---
        def adjust_price(row):
            tkr = row['ticker']
            if tkr in self.ratios_cedear and row['source'] != 'YFinance_USD':
                return row['close_usd'] * self.ratios_cedear[tkr]
            return row['close_usd']
            
        df_px = df_prices_sql.copy()
        df_px['precio_ajustado'] = df_px.apply(adjust_price, axis=1)
        
        df_prices_grid = df_px.pivot_table(index='date', columns='ticker', values='precio_ajustado')
        df_prices_grid = df_prices_grid.reindex(date_range).ffill()

        print("4. Ejecutando Mark-to-Market y Agrupación Barbell...")
        # --- MATRIZ DE VALUACIÓN (M2M) ---
        # Multiplicación tensorial. Las columnas que no existan en precios se asumen 0.
        df_m2m = df_holdings_grid.multiply(df_prices_grid).fillna(0)
        
        # Consolidación de Resultados
        df_metrics = pd.DataFrame(index=date_range)
        df_metrics.index.name = 'Operado'
        
        df_metrics['Cash_Total_USD'] = cash_total_usd
        
        # Agrupación Táctica
        activos_presentes = df_m2m.columns
        safe_cols = [col for col in activos_presentes if col in self.safe_bonds]
        growth_cols = [col for col in activos_presentes if col not in self.safe_bonds]
        
        # Safe Valuation: Bonos Cortos + FCIs Dolarizados + Cauciones Dolarizadas
        df_metrics['Total_Safe_Valuation'] = df_m2m[safe_cols].sum(axis=1) + fci_grid_usd.fillna(0) + cauc_grid_usd.fillna(0)
        
        # Growth Valuation: Acciones, CEDEARs, ETFs
        df_metrics['Total_Growth_Valuation'] = df_m2m[growth_cols].sum(axis=1)
        
        df_metrics['Patrimonio_USD'] = (
            df_metrics['Cash_Total_USD'] + 
            df_metrics['Total_Safe_Valuation'] + 
            df_metrics['Total_Growth_Valuation']
        )
        
        print(f"-> VCP Inicial: US$ {df_metrics['Patrimonio_USD'].iloc[0]:,.2f}")
        print(f"-> VCP Actual:  US$ {df_metrics['Patrimonio_USD'].iloc[-1]:,.2f}")
        return df_metrics.reset_index()

def main():
    csv_path = r'c:\Users\tomas\white_finance\data\analytics\cuentas_unificadas_usd_sorted.csv'
    out_path = r'c:\Users\tomas\white_finance\data\analytics\portfolio_vcp_history.csv'
    
    engine = VectorizedVCPEngine(csv_path)
    df_vcp = engine.build_portfolio_metrics()
    
    df_vcp.to_csv(out_path, index=False)
    print(f"\nProceso Exitoso. Base de datos M2M generada en: {out_path}")

if __name__ == "__main__":
    main()

1. Preprocesando Transacciones y Datos de Mercado...
2. Procesando Componentes Safe Base (Liquidez, FCIs y Cauciones)...
3. Construyendo Matrices Vectorizadas de Equities...
4. Ejecutando Mark-to-Market y Agrupación Barbell...
-> VCP Inicial: US$ 264.54
-> VCP Actual:  US$ 2,532,041.68

Proceso Exitoso. Base de datos M2M generada en: c:\Users\tomas\white_finance\data\analytics\portfolio_vcp_history.csv


Algo está pasando que inicia con 200 dolares y despues termina con dos millones de dolares, o hay que revisar la cuenta corriente historica en dolares ordenada o habrá que ver por qué arranca directamente con 200 dolares, en donde y demás
el otro script que hay que arreglar en todo caso es el que unifica las cuentas corrientes pero creo que ese script ya lo validé.